In [ ]:
!pip install psycopg2
!pip install psycopg2 pandas openpyxl

import pandas as pd
import psycopg2
from datetime import datetime

In [ ]:

host = "ep-blue-rain-a5lv8u2n-pooler.us-east-2.aws.neon.tech"
port = "5432"  # geralmente é 5432
database = "neondb"
user = "neondb_owner"
password = "npg_VMOcWfYoE45U"

In [ ]:
# Função para executar a consulta SQL e retornar os resultados como DataFrame
def execute_query(query):
    try:
        connection = psycopg2.connect(
            host=host,
            port=port,
            dbname=database,
            user=user,
            password=password
        )
        cursor = connection.cursor()
        cursor.execute(query)
        # Recuperar os resultados e colocar em um DataFrame
        columns = [desc[0] for desc in cursor.description]
        rows = cursor.fetchall()
        df = pd.DataFrame(rows, columns=columns)
        cursor.close()
        connection.close()
        return df
    except Exception as e:
        print("Erro ao executar consulta:", e)
        return None

# Obtendo ano e mês atual
current_year_month = datetime.now().strftime("%Y-%m")  # Ex: '2025-01'
first_day_of_month = datetime.now().replace(day=1).strftime("%Y-%m-%d")  # Ex: '2025-01-01'

# Consultas SQL com base no ano e mês atual
queries = {
    "turn_over_mensal": f"""
        SELECT
            COUNT(CASE WHEN hs.situacao_nova = 'Demitido' THEN 1 END) AS total_demitidos,
            COUNT(DISTINCT c.id_contrato) AS total_funcionarios,
            (COUNT(CASE WHEN hs.situacao_nova = 'Demitido' THEN 1 END) * 100.0) /
            NULLIF(COUNT(DISTINCT c.id_contrato), 0) AS taxa_turnover_percent
        FROM
            Contrato c
        LEFT JOIN
            Historico_Situacao_Funcional hs
            ON c.id_contrato = hs.id_contrato
            AND hs.ano_mes_referencia = '{first_day_of_month}'  -- Usando o primeiro dia do mês corrente
        WHERE
            c.ano_mes_referencia = '{first_day_of_month}';  -- Usando o primeiro dia do mês corrente
    """,
    "distribuicao_func_por_departamento": """
        SELECT
            d.nome_departamento,
            COUNT(c.id_contrato) AS total_funcionarios
        FROM
            Departamentos d
        LEFT JOIN
            Contrato c ON d.id_departamento = c.id_departamento
        WHERE
            c.situacao_funcional = 'Ativo'
        GROUP BY
            d.nome_departamento
        ORDER BY
            total_funcionarios DESC;
    """,
    "perfil_demografico": """
        SELECT
            sexo,
            COUNT(id_contrato) AS total,
            ROUND(AVG(EXTRACT(YEAR FROM CURRENT_DATE) - EXTRACT(YEAR FROM data_nascimento)), 1) AS idade_media
        FROM
            Contrato
        WHERE
            situacao_funcional = 'Ativo'
        GROUP BY sexo;
    """
}

# Executando todas as consultas e criando um DataFrame para cada relatório
dfs = {}
for report_name, query in queries.items():
    print(f"Executando consulta para {report_name}...")
    df = execute_query(query)
    if df is not None:
        dfs[report_name] = df

# Salvando os resultados em um arquivo Excel com múltiplas abas
if dfs:
    with pd.ExcelWriter("relatorios_empresa.xlsx", engine="openpyxl") as writer:
        for report_name, df in dfs.items():
            # Renomeando as abas para garantir que o título tenha 31 caracteres ou menos
            safe_report_name = report_name[:31]  # Limita o nome da aba a 31 caracteres
            df.to_excel(writer, sheet_name=safe_report_name, index=False)
    print("Relatórios exportados para 'relatorios_empresa.xlsx'")
else:
    print("Não foi possível gerar os relatórios.")

In [ ]:
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders

# Função para enviar e-mail com anexo
def send_email(subject, body, to_email, attachment_path):
    from_email = "empresa.teste.automatizacao@gmail.com"  # Seu e-mail
    password = "mcmt zqrq lfcw zlev"  # Senha de aplicativo gerada

    # Criando a mensagem MIME
    msg = MIMEMultipart()
    msg['From'] = from_email
    msg['To'] = to_email
    msg['Subject'] = subject

    # Corpo do e-mail
    msg.attach(MIMEText(body, 'plain'))

    # Anexando o arquivo
    part = MIMEBase('application', "octet-stream")
    with open(attachment_path, "rb") as f:
        part.set_payload(f.read())
    encoders.encode_base64(part)
    part.add_header('Content-Disposition', 'attachment; filename="relatorios_empresa_atualizado.xlsx"')
    msg.attach(part)

    # Enviando o e-mail via SMTP
    try:
        # Conexão com o servidor SMTP do Gmail
        server = smtplib.SMTP('smtp.gmail.com', 587)
        server.starttls()  # Criptografia
        server.login(from_email, password)
        server.sendmail(from_email, to_email, msg.as_string())
        server.quit()
        print("E-mail enviado com sucesso!")
    except Exception as e:
        print(f"Erro ao enviar e-mail: {e}")

# Corpo do e-mail
email_body = "Prezado(a),\n\nSegue o relatório de RH gerado para o mês atual.\n\nAtenciosamente,\nEquipe RH"

# Destinatário (e-mail do RH)
rh_email = "gabrielhsmaia@gmail.com"  # Substitua pelo e-mail do RH

# Caminho do arquivo a ser enviado
file_path = "relatorios_empresa.xlsx"

# Enviar o e-mail com o relatório
send_email("Relatório RH - Mês Atual", email_body, rh_email, file_path)
